# Stehekin, WA 2025 Debris Flow Modeling Preparation

Landlab inputs for the Pioneer Fire, WA.

The [landlab contract](https://github.com/gaia-hazlab/landlab_data_prep/blob/master/docs/landlab_workflow_contract.md)'s reference event (`WA4818312053120240608`), in three steps:

1. **Stage** every dataset the landlab workflow reads into one Zarr store, each on its native grid.
2. **Align** the store onto the contract's 10 m UTM grid with `gaia align`, and look at it.
3. **Convert** the aligned store to GeoTIFF and ESRI ASCII grids with GDAL alone ([scripts/landlab-inputs.sh](https://github.com/gaia-hazlab/gaia-cli/blob/main/scripts/landlab-inputs.sh)).

The soil-derived node fields (porosity, cohesion, ...) are not computed here: landlab_data_prep derives them from the seven SOLUS layers.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import rioxarray  # noqa: F401  registers .rio
import xarray as xr
from matplotlib.colors import LightSource, to_rgba
from matplotlib.patches import Patch

from gaia_cli import stations

import cf_xarray as cfxr

from gaia_cli import stations
from gaia_cli.datasets import cf_criteria

# ds.cf["precipitation"]: each source's precipitation (PRISM ppt, Synoptic precip_accum)
cfxr.set_options(custom_criteria=cf_criteria())


#%config InlineBackend.figure_format = 'retina'

## 1. Stage

[examples/pioneer.yaml](https://github.com/gaia-hazlab/gaia-cli/blob/main/examples/pioneer.yaml) lists the required inputs (DEM, burn severity, the seven SOLUS soil layers, STATSGO K-factor, NLCD), plus PRISM and Synoptic precipitation for an atmospheric river in December 2025. The static layers ignore the window. Its AOI, [examples/pioneer-aoi.geojson](https://github.com/gaia-hazlab/gaia-cli/blob/main/examples/pioneer-aoi.geojson), is the perimeter's bounding box, and meets five BAER fires, so `fire` and `year` choose the 2024 Pioneer Fire. The stores and the landlab files go to `/tmp`; `-o` also takes an `s3://` URL, such as the public `gaia-scratch` bucket on Kopah.

In [ ]:
!cat pioneer.yaml

In [ ]:
scratch = "/tmp/gaia-scratch/gaia-cli"  # or an s3:// URL
staged = f"{scratch}/pioneer"                # gaia stage -o
aligned = f"{scratch}/pioneer-10m"           # gaia align -o
staged_zarr = f"{staged}/pioneer.zarr"       # gaia align -i
aligned_zarr = f"{aligned}/pioneer-10m.zarr"
landlab_dir = "/tmp/pioneer-landlab"         # Landlab needs local files

In [ ]:
!gaia stage --config pioneer.yaml -o {staged} --overwrite
# `!` ignores the exit code; IPython keeps it in _exit_code
assert _exit_code == 0, "gaia failed; see its log above"  # noqa: F821

## 2. Align onto the contract grid

`--crs utm --res 10 --aoi` gives the contract's grid: the UTM zone of the AOI (EPSG:32610), 10 m cells, the AOI's bounds snapped outward to multiples of 10 m, and NaN in cells whose centre is outside the AOI. The `-m` options are the contract's resampling methods; without them, DEMs use `bilinear` and categories `mode`. PRISM goes `nearest` (its default, `conservative`, becomes `nearest` onto a finer grid), and Synoptic's stations pass through unchanged.

In [ ]:
!gaia align -i {staged_zarr} -o {aligned} \
    --crs utm --res 10 --aoi pioneer-aoi.geojson \
    -m dem/usgs10m=cubic \
    -m soil/solus100=bilinear \
    -m rasters/nlcd=nearest \
    -m rasters/burn-severity=nearest \
    -m rasters/prism/tmin=bilinear \
    -m rasters/prism/tmax=bilinear \
    --overwrite
# `!` ignores the exit code; IPython keeps it in _exit_code
assert _exit_code == 0, "gaia failed; see its log above"  # noqa: F821

In [ ]:
# A store in the public gaia-scratch bucket on Kopah reads anonymously with obstore
# (S3Store with endpoint="https://s3.kopah.uw.edu", virtual_hosted_style_request=False,
# skip_signature=True, wrapped in zarr.storage.ObjectStore); this one is local
dt = xr.open_datatree(aligned_zarr, engine="zarr", zarr_format=3, consolidated=True)
dt.groups

## A look at the aligned store

In [ ]:
def categories(da, min_share=0.005, by_share=True):
    """RGBA image and legend handles from a categorical variable's CF flag attributes."""
    codes = da.attrs["flag_values"]
    names = da.attrs["flag_meanings"].split()
    colors = da.attrs["flag_colors"].split()
    rgba = np.ones(da.shape + (4,))  # white where there's no class
    handles = []
    for code, name, color in zip(codes, names, colors):
        cells = (da == code).values
        rgba[cells] = to_rgba(color)
        share = cells.sum() / da.notnull().sum().item()
        if share >= min_share:
            handles.append((share, Patch(color=color, label=f"{name.replace('_', ' ')} ({share:.0%})")))
    if by_share:
        handles.sort(key=lambda h: -h[0])
    return rgba, [h for _, h in handles]


elevation = dt["dem/usgs10m"].elevation
soil = dt["soil/solus100"]
prism = dt["rasters/prism"].to_dataset().sel(time=slice("2025-12-01", "2025-12-16"))
precip = prism.cf["precipitation"].sum("time", min_count=1)

# Gauge totals over the same PRISM days (12:00 UTC to 12:00 UTC)
t0, t1 = prism.time_bounds.values[0, 0], prism.time_bounds.values[-1, 1]
hourly = stations.to_grid(dt["stations/synoptic"].to_dataset(), freq="1h", how="sum")
gauge_total = hourly.cf["precipitation"].sel(time=slice(t0 + np.timedelta64(1, "h"), t1)).sum("time", min_count=1)
xy = gpd.GeoSeries(hourly.geometry.values, crs="EPSG:4326").to_crs(precip.rio.crs)

fig, axes = plt.subplots(2, 3, figsize=(15, 11), layout="constrained", sharex=True, sharey=True)
x, y = elevation.x.values, elevation.y.values
extent = [x[0] - 5, x[-1] + 5, y[-1] - 5, y[0] + 5]

ax = axes[0, 0]
shade = LightSource(azdeg=315, altdeg=45).hillshade(elevation.fillna(elevation.min()).values, dx=10, dy=10)
ax.imshow(np.where(elevation.notnull(), shade, np.nan), extent=extent, cmap="gray")
elevation.plot.imshow(ax=ax, cmap="terrain", alpha=0.5, cbar_kwargs={"label": "m (NAVD88)", "shrink": 0.6})
ax.set_title("3DEP 10 m elevation (cubic)")

# burn severity's classes are ordered, NLCD's are listed by area
for ax, da, by_share, title in [
    (axes[0, 1], dt["rasters/burn-severity"].burn_severity, False, "Soil burn severity, Pioneer 2024 (nearest)"),
    (axes[0, 2], dt["rasters/nlcd"].landcover, True, "NLCD 2021 land cover (nearest)"),
]:
    rgba, handles = categories(da, by_share=by_share)
    ax.imshow(rgba, extent=extent)
    ax.legend(handles=handles, loc="lower left", fontsize=7, framealpha=0.9)
    ax.set_title(title)
    # A hidden colorbar takes the same room as the other panels' colorbars, so the grid lines up
    fig.colorbar(plt.cm.ScalarMappable(), ax=ax, shrink=0.6).ax.set_visible(False)

(soil.anylithicdpt / 100).plot.imshow(ax=axes[1, 0], cmap="YlOrBr", cbar_kwargs={"label": "m", "shrink": 0.6})
axes[1, 0].set_title("SOLUS depth to bedrock (bilinear)")
dt["soil/statsgo"].kffact.plot.imshow(ax=axes[1, 1], cmap="viridis", cbar_kwargs={"label": "K-factor", "shrink": 0.6})
axes[1, 1].set_title("STATSGO K-factor (nearest)")

ax = axes[1, 2]
precip.plot.imshow(ax=ax, cmap="Blues", cbar_kwargs={"label": "mm", "shrink": 0.6})
ax.scatter(xy.x, xy.y, c="k", s=25)
for px, py, sid, total in zip(xy.x, xy.y, hourly.station.values, gauge_total.values):
    label = f"{sid} {total:.0f} mm" if np.isfinite(total) else f"{sid} (no reports)"
    ax.annotate(label, (px, py), xytext=(6, -3), textcoords="offset points", fontsize=8,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7, lw=0))
ax.set_title("PRISM precipitation, Dec 1–16 2025, and gauges")

for ax in axes.flat:
    ax.set(xlabel="", ylabel="", aspect="equal", xlim=extent[:2], ylim=extent[2:])
    ax.xaxis.set_major_locator(plt.MaxNLocator(4))

Accumulated precipitation at each gauge, against PRISM in the gauge's cell. PRISM's totals are known at the end of each day (12:00 UTC); hours without a report count as zero.

In [ ]:
fig, axes = plt.subplots(1, hourly.sizes["geometry"], figsize=(15, 3.5), sharey=True, layout="constrained")
ends = prism.time_bounds.values[:, 1]
for i, ax in enumerate(axes):
    gauge = hourly.isel(geometry=i)
    in_window = gauge.cf["precipitation"].sel(time=slice(t0 + np.timedelta64(1, "h"), t1))
    cell = prism.cf["precipitation"].sel(x=xy.x.iloc[i], y=xy.y.iloc[i], method="nearest")
    ax.plot(in_window.time, in_window.fillna(0).cumsum("time"), label="gauge (hourly)")
    ax.plot(np.r_[t0, ends], np.r_[0, cell.cumsum("time").values], marker="o", ms=3, label="PRISM (daily)")
    ax.set_title(f"{gauge.station.item()}, {gauge.elevation.item():.0f} m")
    ax.xaxis.set_major_locator(plt.matplotlib.dates.DayLocator(interval=4))
    ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%b %d"))
axes[0].set_ylabel("accumulated precipitation (mm)")
axes[0].legend(loc="upper left");

## 3. Landlab inputs with GDAL

GDAL reads the aligned store directly: each array is a subdataset with its CRS (`_CRS`), nodata (the Zarr `fill_value`) and, for packed soil layers, its `scale_factor`. A store on S3 is read through `/vsis3/` (or `/vsicurl/`), using its consolidated metadata: for the public `gaia-scratch` bucket, prefix the commands with `AWS_S3_ENDPOINT=s3.kopah.uw.edu AWS_NO_SIGN_REQUEST=YES AWS_VIRTUAL_HOSTING=FALSE` and use `/vsis3/gaia-scratch/...` paths (see [Reading stores with GDAL](https://gaia-hazlab.github.io/gaia-cli/gdal)).

In [ ]:
# Each group's arrays are subdatasets; open them through their group, since GDAL 3.13
# crashes opening the whole store (docs/gdal.md)
!for g in dem/usgs10m rasters/burn-severity rasters/nlcd rasters/prism soil/solus100 soil/statsgo; do \
  gdal raster info "{aligned_zarr}/$$g" 2>/dev/null | grep NAME; done

Since every raster is already on the contract grid, writing landlab's files is a format conversion plus the contract's nodata rules, a few `gdal` commands per layer. [scripts/landlab-inputs.sh](https://github.com/gaia-hazlab/gaia-cli/blob/main/scripts/landlab-inputs.sh) writes a float32 GeoTIFF and an ESRI ASCII grid for each layer:

- nodata (NaN, or a packed integer's fill) becomes `-9999`;
- soil layers are unscaled (`cec7` is stored as tenths, `ph1to1h2o` and `dbovendry` as hundredths);
- burn severity keeps classes 2–4 and sets everything else to 1, with no nodata.

Layers the model reads as they are get their Landlab field names (`topographic__elevation`, `burn__severity`, `landcover`, `kffact`); the soil layers keep SOLUS's names.

In [ ]:
!../scripts/landlab-inputs.sh {aligned_zarr} {landlab_dir}

In [ ]:
!head -6 {landlab_dir}/topographic__elevation.asc | cut -c 1-60

The contract's grid check: every output has the same CRS, transform, shape and nodata.

In [ ]:
rows = []
for tif in sorted(Path(landlab_dir).glob("*.tif")):
    with rasterio.open(tif) as src:
        a = src.read(1, masked=True)
        rows.append({
            "layer": tif.stem, "crs": str(src.crs), "transform": tuple(src.transform)[:6],
            "shape": src.shape, "nodata": "none" if src.nodata is None else src.nodata,
            "min": a.min(), "max": a.max(),
            "nodata cells": int(a.mask.sum()),
        })
check = pd.DataFrame(rows).set_index("layer")
assert check[["crs", "transform", "shape"]].nunique().eq(1).all()
check.drop(columns="transform")